# Clustering Analysis

This notebook performs the clustering analysis using the outputs generated by  '1. extract_dnabert_embeddings.ipynb' and '2. extract_pharokka_features_colab.ipynb' codes.

Specifically, it:
- loads the DNABERT-derived embedding features,
- loads the Pharokka-derived genomic features,
- combines these extracted features into a unified dataset, and
- applies clustering methods to analyze relationships among the sequences.

This notebook assumes that the embedding and feature extraction steps have already been completed and that their output files are available in the expected input locations.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from Bio import Entrez, SeqIO
import time
from tqdm.auto import tqdm
import re
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
import matplotlib.pyplot as plt
import seaborn as sns
import umap.umap_ as umap

## Importing DNABERT embeddings

In [ ]:
df_full = pd.read_csv('embeddings.csv')

## Importing Pharokka features  

This cell imports pharokka features from the extratced outputs of code '2. extract_pharokka_features_colab.ipynb'. Before running the cell, ensure that the outputs are unzipped and the folders name matches PHAROKKA_ROOT  variable.

In [ ]:
PHAROKKA_ROOT = "pharokka_outputs"

def _safe_col(name: str) -> str:
    # make feature names stable and column-safe
    name = str(name).strip()
    name = re.sub(r"\s+", "_", name)
    name = re.sub(r"[^A-Za-z0-9_]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name[:120]  # prevent absurdly long headers

def load_pharokka_features(df: pd.DataFrame) -> pd.DataFrame:

    rows = []

    for _, r in df.iterrows():
        phage = str(r["phage"])
        outdir = Path(PHAROKKA_ROOT) / phage

        feat = {"phage": phage}

        # --- 1) coding density (and optionally pharokka length/gc) ---
        len_tsv = list(outdir.glob("*_length_gc_cds_density.tsv"))
        if len_tsv:
            ldf = pd.read_csv(len_tsv[0], sep="\t")
            # Pharokka output doc: length, gc_perc, translation table, cds_coding_density :contentReference[oaicite:5]{index=5}
            feat["cds_coding_density"] = float(ldf.iloc[0]["cds_coding_density"])
            feat["pharokka_length"] = float(ldf.iloc[0].get("length", np.nan))
            feat["pharokka_gc_perc"] = float(ldf.iloc[0].get("gc_perc", np.nan))
        else:
            feat["cds_coding_density"] = np.nan
            feat["pharokka_length"] = np.nan
            feat["pharokka_gc_perc"] = np.nan

        # --- 2) PHROG categories + counts of tRNAs/tmRNAs/CRISPRs (mobile RNA elements etc.) ---
        # Pharokka docs: cds_functions.tsv includes counts of CDSs, tRNAs, tmRNAs, CRISPRs, and PHROG-assigned functions :contentReference[oaicite:6]{index=6}
        feat["pharokka_tRNA"] = 0.0
        feat["pharokka_tmRNA"] = 0.0
        feat["pharokka_CRISPR"] = 0.0

        func_tsv = list(outdir.glob("*_cds_functions.tsv"))
        if func_tsv:
            fdf = pd.read_csv(func_tsv[0], sep="\t")

            desc_col = None
            cnt_col = None
            for dc in ["Description", "description", "function", "Function", "PHROG", "phrog"]:
                if dc in fdf.columns:
                    desc_col = dc
                    break
            for cc in ["Count", "count", "n", "N"]:
                if cc in fdf.columns:
                    cnt_col = cc
                    break

            if desc_col and cnt_col:
                for desc, cnt in zip(fdf[desc_col], fdf[cnt_col]):
                    if pd.isna(desc) or pd.isna(cnt):
                        continue
                    d = str(desc).strip().lower()
                    c = float(cnt)

                    # capture mobile RNA element counts if present
                    if "trna" == d or "trnas" == d:
                        feat["pharokka_tRNA"] = c
                        continue
                    if "tmrna" == d or "tmrnas" == d:
                        feat["pharokka_tmRNA"] = c
                        continue
                    if "crispr" in d:
                        feat["pharokka_CRISPR"] = c
                        continue

                    # everything else treat as PHROG/functional annotation feature
                    col = "PHROG_" + _safe_col(desc)
                    feat[col] = c

        # --- 3) Virulence factors (VFDB hits) ---
        vf = list(outdir.glob("*_top_hits_vfdb.tsv"))
        if vf:
            vdf = pd.read_csv(vf[0], sep="\t")
            feat["vfdb_hit_count"] = float(len(vdf))
            # try to count unique gene identifiers if present
            gene_col = next((c for c in ["gene", "Gene", "sseqid", "target", "Target", "VFID"] if c in vdf.columns), None)
            feat["vfdb_gene_count"] = float(vdf[gene_col].nunique()) if gene_col else float(len(vdf))
        else:
            feat["vfdb_hit_count"] = 0.0
            feat["vfdb_gene_count"] = 0.0

        # --- 4) Antimicrobial resistance hits (CARD) ---
        card = list(outdir.glob("*_top_hits_card.tsv"))
        if card:
            cdf = pd.read_csv(card[0], sep="\t")
            feat["card_hit_count"] = float(len(cdf))
            gene_col = next((c for c in ["gene", "Gene", "ARO", "ARO_name", "sseqid", "target", "Target"] if c in cdf.columns), None)
            feat["card_gene_count"] = float(cdf[gene_col].nunique()) if gene_col else float(len(cdf))
        else:
            feat["card_hit_count"] = 0.0
            feat["card_gene_count"] = 0.0

        rows.append(feat)

    phar_df = pd.DataFrame(rows)

    out = df.merge(phar_df, on="phage", how="left", suffixes=("", "_phar"))

    if "cds_coding_density_phar" in out.columns:
        out["cds_coding_density"] = out["cds_coding_density_phar"]
        out = out.drop(columns=["cds_coding_density_phar"])

    count_like_prefixes = ("PHROG_", "vfdb_", "card_", "pharokka_tRNA", "pharokka_tmRNA", "pharokka_CRISPR")
    for col in out.columns:
        if col.startswith(count_like_prefixes):
            out[col] = out[col].fillna(0.0)

    return out

In [ ]:
df_full = load_pharokka_features(df_full)

In [ ]:
df_full.describe().T

## Droping zero columns and unnecessary pharokka features

In [ ]:
zero_all = df_full.select_dtypes(include="number").columns[(df_full.select_dtypes(include="number") == 0).all()]
zero_all.tolist(), len(zero_all)

num = df_full.select_dtypes(include="number")
zero_all_cols = num.columns[(num == 0).all()]

df_full = df_full.drop(columns=zero_all_cols)
print("Dropped:", len(zero_all_cols))


pharokka_unncessary_columns = [ "PHROG_other",  "PHROG_unknown_function",  "PHROG_lysis",
                               "pharokka_length", "pharokka_gc_perc"]

df_full = df_full.drop(columns=pharokka_unncessary_columns)

## Different feature-combination performance evaluation  
Uncomment the following four cells to change the combinations and their effects on the clustering results

In [ ]:
# pharokka_features_columns = [ "pharokka_tRNA", "PHROG_DNA_RNA_and_nucleotide_metabolism", "PHROG_integration_and_excision", 
#                                  "PHROG_transcription_regulation",  "PHROG_CDS", "PHROG_connector" , "PHROG_head_and_packaging", 
#                                  "PHROG_moron_auxiliary_metabolic_gene_and_host_takeover",  "PHROG_tail"]

# df_full = df_full.drop(columns=pharokka_features_columns)

In [ ]:
# dnabert_columns = [c for c in df_full.columns if c.startswith("emb")]
# df_full = df_full.drop(columns= dnabert_columns)

In [ ]:
# df_full = df_full.drop(columns=[ "cds_coding_density"])

In [ ]:
# df_full = df_full.drop(columns=["length", "gc_content", "cds_coding_density"])

In [ ]:
df_full.describe().T

## Clustering

In [ ]:

OUTDIR = Path("clustering results")
OUTDIR.mkdir(parents=True, exist_ok=True)


n_umap_cluster = 2
umap_neighbors = 10
umap_min_dist = 0.4

# 2D UMAP only for plotting/exporting coordinates
umap2d_neighbors = 10
umap2d_min_dist = 0.4

k_values = [1, 2, 3, 4, 5]
random_state = 42
n_init = 10


df = df_full.copy()

# Optional: if you have an ID column, keep it; otherwise we index by row
id_col = ["phage" , "Unnamed: 0", "ncbi_acc"]

feat_df = df.drop(columns=id_col)
feat_df = feat_df.apply(pd.to_numeric, errors="coerce").fillna(0.0)

X = feat_df.values
X_scaled = StandardScaler().fit_transform(X)

print("Feature matrix shape:", X_scaled.shape)


umap_cluster = umap.UMAP(
    n_components=n_umap_cluster,
    n_neighbors=umap_neighbors,
    min_dist=umap_min_dist,
    random_state=random_state
)
X_umap_n = umap_cluster.fit_transform(X_scaled)
print("UMAP clustering embedding shape:", X_umap_n.shape)

for j in range(n_umap_cluster):
    df[f"umap_nd_{j+1}"] = X_umap_n[:, j]


metrics = []
labels_by_k = {}

for k in k_values:
    km = KMeans(n_clusters=k, random_state=random_state, n_init=n_init)
    labels = km.fit_predict(X_umap_n)
    labels_by_k[k] = labels

    if k == 1:
        sil = np.nan
        dbi = np.nan
    else:
        sil = silhouette_score(X_umap_n, labels)
        dbi = davies_bouldin_score(X_umap_n, labels)

    metrics.append({
        "k": k,
        "inertia": km.inertia_,
        "silhouette": sil,
        "davies_bouldin": dbi
    })

metrics_df = pd.DataFrame(metrics).sort_values("k").reset_index(drop=True)
print(metrics_df)


# 2D UMAP for plotting/export

umap_2d = umap.UMAP(
    n_components=2,
    n_neighbors=umap2d_neighbors,
    min_dist=umap2d_min_dist,
    random_state=random_state
)
xy = umap_2d.fit_transform(X_scaled)

df["umap_1"] = xy[:, 0]
df["umap_2"] = xy[:, 1]


sns.set(style="whitegrid")

k_plot_values = [k for k in k_values if k >= 2]

fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
axes = axes.ravel()

for ax, k in zip(axes, k_plot_values):
    tmp = df.copy()
    tmp["cluster"] = labels_by_k[k]

    sns.scatterplot(
        ax=ax,
        data=tmp,
        x="umap_1",
        y="umap_2",
        hue="cluster",
        palette="viridis",
        s=90,
        alpha=0.95,
        edgecolor=None
    )
    ax.set_title(f"KMeans (UMAP→{n_umap_cluster}D), k={k}")
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.legend(title="Cluster", loc="best")

fig.savefig(OUTDIR / "kmeans_umap2d_subplots_k2_k5.png", dpi=1200, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(metrics_df["k"], metrics_df["inertia"], marker="o")
plt.title("Elbow Method (Inertia vs k)")
plt.xlabel("k")
plt.ylabel("Inertia")
plt.xticks(k_values)
plt.grid(True, linestyle="--", alpha=0.5)
plt.savefig(OUTDIR / "elbow_k1_to_k5.png", dpi=1200, bbox_inches="tight")
plt.show()

score_df = metrics_df[metrics_df["k"] >= 2].copy()
plt.figure(figsize=(8, 5))
plt.plot(score_df["k"], score_df["silhouette"], marker="o", label="Silhouette (higher better)")
plt.plot(score_df["k"], score_df["davies_bouldin"], marker="o", label="Davies–Bouldin (lower better)")
plt.title("Clustering Metrics vs k (k≥2)")
plt.xlabel("k")
plt.ylabel("Score")
plt.xticks(score_df["k"].tolist())
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.savefig(OUTDIR / "scores_k2_to_k5.png", dpi=1200, bbox_inches="tight")
plt.show()


# Export assignments + full df with labels/coords

if id_col is not None:
    assign_df = pd.DataFrame({id_col[0]: df[id_col[0]].astype(str)}).set_index(id_col[0])
else:
    assign_df = pd.DataFrame(index=df.index)

for k in k_values:
    assign_df[f"k={k}"] = labels_by_k[k].astype(int)

out_xlsx = OUTDIR / f"UMAP{n_umap_cluster}D_k1_to_k5.xlsx"
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    assign_df.to_excel(writer, sheet_name="cluster_assignments")
    metrics_df.to_excel(writer, sheet_name="metrics", index=False)

    df_with_labels = df.copy()
    for k in k_values:
        df_with_labels[f"cluster_k{k}"] = labels_by_k[k].astype(int)

    df_with_labels.to_excel(writer, sheet_name="df_full_with_labels", index=False)

print(f"\nWrote: {out_xlsx}")